In [1]:
import os
import sys
from google.colab import drive

# 1. Mount Drive (Force remount to prevent Transport Endpoint errors)
drive.mount('/content/drive', force_remount=True)

# 2. Strictly enforce working directory to target the real repository
PROJECT_DIR = '/content/drive/MyDrive/GalaxEye Space — Technical Assessment Submission/open-cd'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f"Current Working Directory: {os.getcwd()}")

# 3. FORCE APPLY PATCHES (Hardcoded to Colab's default dist-packages)
site_packages = '/usr/local/lib/python3.12/dist-packages'

# A. Patch mmcv version checks
for lib in ['mmseg', 'mmdet', 'opencd']:
    path = "opencd/__init__.py" if lib == 'opencd' else os.path.join(site_packages, lib, "__init__.py")
    if os.path.exists(path):
        with open(path, 'r') as f: content = f.read()
        content = content.replace("'2.2.0'", "'2.3.0'").replace('"2.2.0"', '"2.3.0"')
        with open(path, 'w') as f: f.write(content)

# B. Apply the mmpretrain BLIP 'NoneType' bug patch
blip_path = os.path.join(site_packages, "mmpretrain", "models", "multimodal", "blip", "language_model.py")
if os.path.exists(blip_path):
    with open(blip_path, 'r') as f: content = f.read()
    content = content.replace('PreTrainedModel = None', 'PreTrainedModel = object')
    with open(blip_path, 'w') as f: f.write(content)

# 4. Check if Colab environment needs restoration
try:
    import mmengine
    import mmcv
    import transformers
    import rasterio
    print("\n✅ Environment dependencies and patches are intact. Proceed to Cell 2!")
except ImportError:
    print("\n⚠️ New Colab session detected. Restoring OpenMMLab dependencies...")
    p_host = "download" + "." + "pytorch" + ".org"
    pt_url = f"https://{p_host}/whl/cu121"
    m_host = "download" + "." + "openmmlab" + ".com"
    mmcv_url = f"https://{m_host}/mmcv/dist/cu121/torch2.3.0/index.html"

    !pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url {pt_url}
    !pip install --upgrade pip setuptools wheel ninja
    !pip install -U mmengine "mmpretrain>=1.0.0rc7"
    !pip install mmcv==2.2.0 -f {mmcv_url}
    !pip install "mmsegmentation>=1.2.2" "mmdet>=3.0.0" ftfy regex "transformers<4.36.0"
    !pip install rasterio tifffile albumentations
    !pip install -v -e .

    print("\n=======================================================================")
    print("🛑 CRITICAL: CLICK 'RUNTIME > RESTART SESSION' IN COLAB NOW! 🛑")
    print("=======================================================================")
    print("After restarting, run this Cell 1 one more time. It will print the '✅' message.")

Mounted at /content/drive
Current Working Directory: /content/drive/MyDrive/GalaxEye Space — Technical Assessment Submission/open-cd


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(



✅ Environment dependencies and patches are intact. Proceed to Cell 2!


In [ ]:
import os
import sys
import torch
import warnings

PROJECT_DIR = '/content/drive/MyDrive/GalaxEye Space — Technical Assessment Submission/open-cd'
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# ==============================================================================
# 1. PHASE 3: BUILD SIAMESE BACKBONE
# ==============================================================================
os.makedirs('opencd/models/backbones', exist_ok=True)
backbone_code = """import torch
import torch.nn as nn
from mmseg.registry import MODELS

@MODELS.register_module()
class PseudoSiamChangeFormer(nn.Module):
    def __init__(self, backbone_cfg, in_channels_eo=3, in_channels_sar=1):
        super().__init__()
        cfg_eo = backbone_cfg.copy()
        cfg_eo['in_channels'] = in_channels_eo
        self.encoder_eo = MODELS.build(cfg_eo)

        cfg_sar = backbone_cfg.copy()
        cfg_sar['in_channels'] = in_channels_sar
        self.encoder_sar = MODELS.build(cfg_sar)

        if hasattr(self.encoder_eo, 'layers'):
            for i in range(len(self.encoder_eo.layers)):
                self.encoder_sar.layers[i][1] = self.encoder_eo.layers[i][1]
                self.encoder_sar.layers[i][2] = self.encoder_eo.layers[i][2]
                if i > 0:
                    self.encoder_sar.layers[i][0] = self.encoder_eo.layers[i][0]
        else:
            raise NotImplementedError("This wrapper targets mmseg.MixVisionTransformer.")

    def forward(self, x):
        eo = x[:, :3, :, :]
        sar = x[:, 3:, :, :]
        out_eo = self.encoder_eo(eo)
        out_sar = self.encoder_sar(sar)
        return tuple([torch.cat([e, s], dim=1) for e, s in zip(out_eo, out_sar)])
"""
with open('opencd/models/backbones/pseudo_siam.py', 'w') as f: f.write(backbone_code)

init_path = 'opencd/models/backbones/__init__.py'
if not os.path.exists(init_path): open(init_path, 'w').close()
with open(init_path, 'r') as f: content = f.read()
if 'from .pseudo_siam import PseudoSiamChangeFormer' not in content:
    with open(init_path, 'a') as f: f.write("\nfrom .pseudo_siam import PseudoSiamChangeFormer\n")

print("✅ Phase 3 Backbone Generated.")

# ==============================================================================
# 2. PHASE 4: BUILD MASTER CONFIGURATION (GPU Optimized)
# ==============================================================================
os.makedirs('configs/changeformer', exist_ok=True)
config_code = """_base_ = [
    '../_base_/datasets/disaster_hetero_cd.py',
    '../_base_/default_runtime.py'
]

custom_imports = dict(imports=['opencd.models.backbones.pseudo_siam'], allow_failed_imports=False)

model = dict(
    type='mmseg.EncoderDecoder',
    data_preprocessor=dict(
        type='mmseg.SegDataPreProcessor',
        mean=[123.675, 116.28, 103.53, 127.5],
        std=[58.395, 57.12, 57.375, 57.12],
        bgr_to_rgb=False,
        pad_val=0,
        seg_pad_val=255,
        size_divisor=32),

    backbone=dict(
        type='PseudoSiamChangeFormer',
        in_channels_eo=3,
        in_channels_sar=1,
        backbone_cfg=dict(
            type='mmseg.MixVisionTransformer',
            in_channels=3,
            embed_dims=64,
            num_stages=4,
            num_layers=[3, 4, 18, 3],
            num_heads=[1, 2, 5, 8],
            patch_sizes=[7, 3, 3, 3],
            sr_ratios=[8, 4, 2, 1],
            out_indices=(0, 1, 2, 3),
            mlp_ratio=4,
            qkv_bias=True,
            drop_rate=0.0,
            attn_drop_rate=0.0,
            drop_path_rate=0.1)),

    decode_head=dict(
        type='mmseg.SegformerHead',
        in_channels=[128, 256, 640, 1024],
        in_index=[0, 1, 2, 3],
        channels=256,
        dropout_ratio=0.1,
        num_classes=2,
        norm_cfg=dict(type='BN', requires_grad=True),
        align_corners=False,
        loss_decode=[
            dict(type='mmseg.CrossEntropyLoss', use_sigmoid=False, class_weight=[0.016, 0.984], loss_weight=1.0, loss_name='loss_ce'),
            dict(type='mmseg.DiceLoss', loss_weight=1.0, loss_name='loss_dice')
        ]),
    train_cfg=dict(),
    test_cfg=dict(mode='whole'))

optim_wrapper = dict(
    type='mmengine.OptimWrapper',
    optimizer=dict(type='AdamW', lr=0.0001, weight_decay=0.01),
    paramwise_cfg=dict(
        custom_keys={'pos_block': dict(decay_mult=0.), 'norm': dict(decay_mult=0.), 'head': dict(lr_mult=10.)}))

param_scheduler = [
    dict(type='LinearLR', start_factor=1e-6, by_epoch=False, begin=0, end=500),
    dict(type='CosineAnnealingLR', begin=500, by_epoch=False, T_max=4500, eta_min=0.0)
]

train_cfg = dict(type='IterBasedTrainLoop', max_iters=5000, val_interval=500)
val_cfg = dict(type='ValLoop')
test_cfg = dict(type='TestLoop')

default_hooks = dict(
    checkpoint=dict(type='CheckpointHook', by_epoch=False, interval=500, max_keep_ckpts=3, save_best='mIoU'),
    logger=dict(type='LoggerHook', interval=50, log_metric_by_epoch=False)
)
"""
with open('configs/changeformer/custom_disaster.py', 'w') as f: f.write(config_code)
print("✅ Phase 4 Configuration Generated.")

# ==============================================================================
# 3. DRY-RUN MATHEMATICAL VERIFICATION
# ==============================================================================
warnings.filterwarnings('ignore')

# Clear old cached modules
for mod in list(sys.modules.keys()):
    if 'opencd' in mod: del sys.modules[mod]

from mmengine.config import Config
from mmseg.registry import MODELS
from opencd.models.backbones.pseudo_siam import PseudoSiamChangeFormer

print("\n--- Architecture Compilation Check ---")
try:
    cfg = Config.fromfile('configs/changeformer/custom_disaster.py')
    model = MODELS.build(cfg.model)
    model.eval()

    dummy_input = torch.randn(1, 4, 256, 256)
    with torch.no_grad():
        output = model.extract_feat(dummy_input)

    print("✅ Model Built Successfully!")
    print("✅ Forward Pass Succeeded!")
    print("\n--- Extracted Multi-Scale Concatenated Features ---")
    for i, feat in enumerate(output):
        print(f"Stage {i+1} Shape: {feat.shape}")

    print("\n🎯 SUCCESS! Architecture is mathematically verified and ready for Phase 5.")
except Exception as e:
    import traceback
    print(f"❌ Error during compilation:")
    traceback.print_exc()

✅ Phase 3 Backbone Generated.
✅ Phase 4 Configuration Generated.
